In [ ]:
import pandas as pd


input_df = pd.read_excel("spacy-corrected.xlsx")

,sentence_id,word,tag
0,1,Setting,B-EVENT
1,1,new,I-EVENT
2,1,Return-To-Home,I-EVENT
3,1,altitude,I-EVENT
4,1,to,I-EVENT
...,...,...,...
6201,504,safety,I-MITIGATION
6202,504,during,I-MITIGATION
6203,504,the,I-MITIGATION
6204,504,flight,E-MITIGATION


In [ ]:
import pandas as pd
def fix_code(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Fix the tags for the (Code: XXXX) patterns
    """
    sents = dataframe['sentence_id'].to_list()
    words = dataframe['word'].to_list()
    tags = dataframe['tag'].to_list()
    i = 0
    while i < len(words):
        # Look for the start of a potential block
        if words[i].startswith('(Code') and tags[i] == 'O':
            # print(f"({sents[i]}{words[i]}-{tags[i]})")
            tags[i] = 'B-CONTEXT'
            tags[i+1] = 'I-CONTEXT'
            tags[i+2] = 'E-CONTEXT'
        if words[i].startswith('(code') and tags[i] == 'O':
            # print(f"({sents[i]}{words[i]}-{tags[i]})")
            tags[i] = 'B-CONTEXT'
            tags[i+1] = 'I-CONTEXT'
            tags[i+2] = 'E-CONTEXT'
        i += 1

    return pd.DataFrame({
        'sentence_id': sents,
        'word': words,
        'tag': tags
    })

In [15]:
code_fixed = fix_code(input_df)

(6(Code-O)
(13(Code-O)
(15(Code-O)
(20(Code-O)
(22(Code-O)
(23(Code-O)
(24(Code-O)
(25(Code-O)
(26(Code-O)
(28(Code-O)
(32(Code-O)
(35(Code-O)
(37(Code-O)
(40(Code-O)
(43(Code-O)
(44(Code-O)
(45(Code-O)
(46(Code-O)
(47(Code-O)
(48(Code-O)
(49(Code-O)
(51(Code-O)
(57(Code-O)
(58(Code-O)
(60(Code-O)
(62(Code-O)
(64(Code-O)
(65(Code-O)
(72(Code-O)
(73(Code-O)
(77(Code-O)
(83(Code-O)
(89(Code-O)
(98(Code-O)
(100(Code-O)
(101(Code-O)
(103(Code-O)
(104(Code-O)
(107(Code-O)
(108(Code-O)
(113(Code-O)
(114(Code-O)
(115(Code-O)
(117(Code-O)
(120(Code-O)
(121(Code-O)
(122(Code-O)
(125(Code-O)
(127(Code-O)
(128(Code-O)
(129(Code-O)
(131(Code-O)
(132(Code-O)
(135(Code-O)
(137(Code-O)
(138(Code-O)
(139(Code-O)
(143(Code-O)
(144(Code-O)
(145(Code-O)
(147(Code-O)
(153(Code-O)
(158(Code-O)
(164(Code-O)
(167(Code-O)
(168(Code-O)
(170(Code-O)
(173(Code-O)
(174(Code-O)
(178(Code-O)
(180(Code-O)
(188(Code-O)
(189(Code-O)
(190(Code-O)
(194(Code-O)
(195(Code-O)
(199(Code-O)
(200(Code-O)
(203(Code-O)
(204(Cod

In [ ]:
def hanlde_edge(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Fix the tags for the (Statement) patterns 
    that are assigned O tag at the beginning and the end of the span
    """
    sents = dataframe['sentence_id'].to_list()
    words = dataframe['word'].to_list()
    tags = dataframe['tag'].to_list()
    i = 0
    while i < len(words):
        # Look for the start of a potential block
        if words[i].startswith('(') and tags[i] == 'O':
            start_idx = i
            end_idx = -1
            
            # Look for the corresponding end
            for j in range(i, len(words)):
                if words[j].endswith(')'):
                    end_idx = j
                    break

        i += 1

In [22]:
# code_fixed.to_excel('code_fixed.xlsx')
hanlde_edge(code_fixed)

(11(Airport-O)
(33(0x1610008f)-O)
(72(Altitude-O)
(218(Altitude-O)
(423(lost-O)
(424(0x16080029)-O)


In [ ]:
import pandas as pd
from typing import List

def correct_sentence_tags(sentence_df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies correction logic for parenthesis-related tagging errors within a single sentence.
    
    Args:
        sentence_df: A DataFrame containing the data for one sentence_id.
        
    Returns:
        A DataFrame with corrected tags.
    """
    words = sentence_df['word'].tolist()
    tags = sentence_df['tag'].tolist()
    
    # --- Fix Case 2 First: Create new CONTEXT entities for O-tagged parenthetical blocks ---
    # Example: (Code : 180059)
    i = 0
    while i < len(words):
        # Look for the start of a potential block
        if words[i].startswith('(') and tags[i] == 'O':
            start_idx = i
            end_idx = -1
            
            # Look for the corresponding end
            for j in range(i, len(words)):
                if words[j].endswith(')'):
                    end_idx = j
                    break
            
            # If we found a start and an end
            if end_idx != -1:
                # Check if all tags in between are 'O'
                # is_all_O = all(tags[k] == 'O' for k in range(start_idx, end_idx + 1))
                
                # if is_all_O:
                    # Apply BIOES tagging to the block
                if start_idx == end_idx: # Single-token entity like "(Code)"
                    tags[start_idx] = 'S-CONTEXT'
                else: # Multi-token entity like "(Code : 180059)"
                    tags[start_idx] = 'B-CONTEXT'
                    tags[end_idx] = 'E-CONTEXT'
                    for k in range(start_idx + 1, end_idx):
                        tags[k] = 'I-CONTEXT'
                    
                # Skip the index to the end of the block we just tagged
                i = end_idx
        i += 1
        
    # --- Fix Case 1 Second: Extend existing entities to include adjacent parentheses ---
    # Example: Zone E-EVENT, (Airport O -> Zone I-EVENT, (Airport B-CONTEXT
    # Example: Power E-CONTEXT, Plant) O -> Power I-CONTEXT, Plant) E-CONTEXT
    for i in range(len(words)):
        # Extend to the right (absorb an opening parenthesis)
        if words[i].startswith('(') and tags[i] == 'O':
            if i + 1 < len(words) and tags[i+1].startswith('B-'):
                entity_type = tags[i+1][2:] # Get entity type like 'CONTEXT'
                tags[i] = f'B-{entity_type}'
                tags[i+1] = f'I-{entity_type}'

        # Extend to the left (absorb a closing parenthesis)
        if words[i].endswith(')') and tags[i] == 'O':
            if i > 0 and tags[i-1] != 'O':
                # Get entity type from the previous tag
                entity_type = tags[i-1][2:]
                tags[i] = f'E-{entity_type}'
                
                # Downgrade the previous tag if it was an S- or E-
                if tags[i-1].startswith('S-'):
                    tags[i-1] = f'B-{entity_type}'
                elif tags[i-1].startswith('E-'):
                    tags[i-1] = f'I-{entity_type}'

        if words[i].startswith('(') and words[i].endswith(')') and tags[i] == 'O':
            tags[i] = 'S-CONTEXT'

    sentence_df['tag'] = tags
    return sentence_df

def fix_parenthesis_issues(input_excel_path: str, output_excel_path: str):
    """
    Reads an Excel file, corrects parenthesis-related tagging issues,
    and saves the result to a new Excel file.
    
    Args:
        input_excel_path: Path to the source Excel file.
        output_excel_path: Path to save the corrected Excel file.
    """
    print(f"Reading data from '{input_excel_path}'...")
    df = pd.read_excel(input_excel_path)
    
    print("Applying corrections for parenthesis edge cases...")
    # Group by sentence, apply the correction function, and combine the results
    corrected_df = df.groupby('sentence_id', group_keys=False).apply(correct_sentence_tags)
    
    print(f"Saving corrected data to '{output_excel_path}'...")
    corrected_df.to_excel(output_excel_path, index=False)
    print("✅ Done.")

# --- How to Use ---
if __name__ == '__main__':
    # Define your input and output file names
    # Make sure 'your_annotations.xlsx' exists and is in the same directory
    # as this script, or provide the full path.
    input_file = 'spacy-corrected.xlsx' 
    output_file = 'spacy-final-fixed.xlsx'
    
    # As a demonstration, we will create a dummy input file first.
    # In your real use case, you can comment out or delete this block.
    print("--- Creating a dummy input file for demonstration ---")
    # dummy_data = {
    #     'sentence_id': [11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6],
    #     'word': ['GEO', ':', 'You', 'are', 'in', 'a', 'Warning', 'Zone', '(Airport', 'Class', 'Airspace', 'Unpaved', 'Airports', 'Power', 'Plant)', '.', 'Fly', 'with', 'caution', '.', 'Aircraft', 'not', 'in', 'flight', '.', 'Vision', 'systems', 'and', 'obstacle', 'avoidance', 'disabled', '(Code', ':', '180059)', '.'],
    #     'tag': ['S-CONTEXT', 'O', 'B-EVENT', 'I-EVENT', 'I-EVENT', 'I-EVENT', 'I-EVENT', 'E-EVENT', 'O', 'B-CONTEXT', 'I-CONTEXT', 'I-CONTEXT', 'I-CONTEXT', 'E-CONTEXT', 'O', 'O', 'B-MITIGATION', 'I-MITIGATION', 'E-MITIGATION', 'O', 'B-EVENT', 'I-EVENT', 'I-EVENT', 'E-EVENT', 'O', 'B-EVENT', 'I-EVENT', 'I-EVENT', 'I-EVENT', 'I-EVENT', 'E-EVENT', 'O', 'O', 'O', 'O']
    # }
    # pd.DataFrame(dummy_data).to_excel(input_file, index=False)
    # print(f"Dummy file '{input_file}' created.\n")
    
    # Run the main correction function
    fix_parenthesis_issues(input_file, output_file)

--- Creating a dummy input file for demonstration ---
Reading data from 'spacy-corrected.xlsx'...
Applying corrections for parenthesis edge cases...
Saving corrected data to 'spacy-final-fixed.xlsx'...


C:\Users\sward\AppData\Local\Temp\ipykernel_24248\1899703183.py:95: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  corrected_df = df.groupby('sentence_id', group_keys=False).apply(correct_sentence_tags)


✅ Done.
